<a href="https://colab.research.google.com/github/faisu6339-glitch/LLMs/blob/main/Self_Attention(Advanced).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Yes. The most important thing to understand is that **Q, K, V are not three different inputs** in self-attention. They are three different **learned projections of the same input sequence**.

## 1. Why do we use Q, K, V?

Suppose our sentence is:

```text
"The cat sat on the mat"
```

For every word, we have an embedding:

```text
Input embeddings
       ↓
 ┌─────────────────┐
 │ The │ cat │ sat │ on │ the │ mat │
 └─────────────────┘
       ↓
   X embeddings
```

Self-attention needs to answer three questions:

1.  **Query (Q):** What information is this word looking for?
2.  **Key (K):** What information does each word offer for matching?
3.  **Value (V):** What actual information should be passed forward?

So:

```text
X
│
├── XWQ ──→ Q (Query)
│
├── XWK ──→ K (Key)
│
└── XWV ──→ V (Value)
```

The matrices `WQ`, `WK`, and `WV` are **learnable parameters**.

---

# 2. Simple real-world analogy

Imagine a library.

You ask:

> "I want books about machine learning."

Your **Query** is:

```text
"What am I looking for?"
```

Every book has a **Key** describing what it contains:

```text
Book 1 → Python
Book 2 → Machine Learning
Book 3 → Cooking
Book 4 → Deep Learning
```

You compare:

```text
Query ↔ Keys
```

and find that Book 2 and Book 4 are highly relevant.

Then you actually retrieve the **Value** — the information/content from those books.

So:

```text
Q = What am I looking for?
K = What does each item represent?
V = What information should I retrieve?
```

---

# 3. Why can't we just use X directly?

This is an excellent question.

Suppose:

```text
X = input embeddings
```

If we simply calculated:

```text
X × Xᵀ
```

we would have much less flexibility.

Instead, Transformer learns three different representations:

```text
Q = XWQ
K = XWK
V = XWV
```

This allows the model to learn:

*   what information is important for **asking**
*   what information is important for **matching**
*   what information is important for **retrieving**

This separation makes attention much more powerful.

---

# 4. Complete mathematical formula

The fundamental self-attention equation is:

$$Attention(Q,K,V)
=
softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

There are four major steps.

### Step 1 — Create Q, K, V

$$Q=XW_Q$$

$$K=XW_K$$

$$V=XW_V$$

### Step 2 — Calculate similarity

$$Scores=QK^T$$

### Step 3 — Scale

$$Scores=\frac{QK^T}{\sqrt{d_k}}$$

### Step 4 — Softmax

$$AttentionWeights=
softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)$$

### Step 5 — Weighted sum of V

$$Output=AttentionWeights\times V$$

The complete flow:

```text
                 Input X
                    │
          ┌─────────┼─────────┐
          ↓         ↓         ↓
       XWQ        XWK        XWV
          ↓         ↓         ↓
          Q         K         V
          │         │
          └────┬────┘
               ↓
             QKᵀ
               ↓
          divide √dk
               ↓
            Softmax
               ↓
       Attention Weights
               │
               ↓
        Weights × V
               ↓
          Attention
           Output
```

---

# 5. Full program from scratch — no PyTorch

Let's first implement self-attention using **NumPy only**.

This is the best program to understand what's actually happening.

In [1]:
import numpy as np

# ------------------------------------------------
# 1. Input
# ------------------------------------------------

X = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 2.0],
    [1.0, 1.0, 1.0, 1.0]
])

print("Input X:")
print(X)
print("Shape:", X.shape)


# ------------------------------------------------
# 2. Weight matrices
# ------------------------------------------------

W_Q = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [0.0, 1.0]
])

W_K = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [0.0, 1.0]
])

W_V = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.5, 0.5],
    [0.5, 0.5]
])


# ------------------------------------------------
# 3. Create Q, K, V
# ------------------------------------------------

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("\nQ:")
print(Q)

print("\nK:")
print(K)

print("\nV:")
print(V)


# ------------------------------------------------
# 4. Calculate QK^T
# ------------------------------------------------

scores = Q @ K.T

print("\nRaw Attention Scores:")
print(scores)


# ------------------------------------------------
# 5. Scale scores
# ------------------------------------------------

d_k = K.shape[1]

scaled_scores = scores / np.sqrt(d_k)

print("\nScaled Scores:")
print(scaled_scores)


# ------------------------------------------------
# 6. Softmax
# ------------------------------------------------

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


attention_weights = softmax(scaled_scores)

print("\nAttention Weights:")
print(attention_weights)


# ------------------------------------------------
# 7. Weighted sum of V
# ------------------------------------------------

output = attention_weights @ V

print("\nFinal Self-Attention Output:")
print(output)

Input X:
[[1. 0. 1. 0.]
 [0. 2. 0. 2.]
 [1. 1. 1. 1.]]
Shape: (3, 4)

Q:
[[2. 0.]
 [0. 4.]
 [2. 2.]]

K:
[[2. 0.]
 [0. 4.]
 [2. 2.]]

V:
[[1.5 0.5]
 [1.  3. ]
 [2.  2. ]]

Raw Attention Scores:
[[ 4.  0.  4.]
 [ 0. 16.  8.]
 [ 4.  8.  8.]]

Scaled Scores:
[[ 2.82842712  0.          2.82842712]
 [ 0.         11.3137085   5.65685425]
 [ 2.82842712  5.65685425  5.65685425]]

Attention Weights:
[[4.85647715e-01 2.87045707e-02 4.85647715e-01]
 [1.21618317e-05 9.96506553e-01 3.48128496e-03]
 [2.87045707e-02 4.85647715e-01 4.85647715e-01]]

Final Self-Attention Output:
[[1.72847157 1.300233  ]
 [1.00348737 2.99648831]
 [1.5        2.44259086]]


---

# 6. What happens internally?

Suppose we have:

```text
X
↓
Q K V
```

For example:

```text
Q =
[[1 0]
 [0 2]
 [2 2]]
```

and:

```text
K =
[[1 0]
 [0 2]
 [2 2]]
```

Now:

$$QK^T$$

produces:

```text
       K1  K2  K3

Q1     1   0   2
Q2     0   4   4
Q3     2   4   8
```

These numbers represent **how strongly one token matches another token**.

For example:

```text
Q3 · K3 = 8
```

means token 3 has a strong similarity with token 3.

---

# 7. Why transpose K?

This part is very important.

Suppose:

```text
Q shape = (sequence_length, d_k)

K shape = (sequence_length, d_k)
```

We need:

```text
Q × Kᵀ
```

because:

```text
(sequence_length × d_k)
        ×
(d_k × sequence_length)
```

gives:

```text
sequence_length × sequence_length
```

For 5 tokens:

```text
Q       Kᵀ

5×d     d×5

   ↓

    5×5 attention matrix
```

So we get a score for **every token against every other token**.

---

# 8. Full example with words

Let's use:

```text
"The cat sat"
```

Imagine:

```text
The → [0.2, 0.1, 0.3, 0.4]
cat → [0.8, 0.2, 0.1, 0.3]
sat → [0.1, 0.7, 0.2, 0.2]
```

These are the input embeddings:

```text
X
│
├── The
├── cat
└── sat
```

We create:

```text
Q = XWQ

K = XWK

V = XWV
```

Now suppose for `"sat"`:

```text
Q_sat
```

is compared with:

```text
K_The
K_cat
K_sat
```

We might get:

```text
             The     cat     sat
sat Query   0.2     2.1     1.4
```

After softmax:

```text
             The     cat     sat
             0.08    0.57    0.35
```

This means:

```text
sat
 │
 ├── 8% attention → The
 ├── 57% attention → cat
 └── 35% attention → sat
```

Then:

$$Output_{sat}
=
0.08V_{The}
+
0.57V_{cat}
+
0.35V_{sat}$$

That's the core idea of self-attention.

---

# 9. Program with actual words

Here is a more understandable educational implementation.

In [2]:
import numpy as np

# -----------------------------------------
# Sentence
# -----------------------------------------

words = ["The", "cat", "sat"]

# Simple embeddings
X = np.array([
    [1, 0, 1, 0],   # The
    [0, 1, 0, 1],   # cat
    [1, 1, 1, 1]    # sat
], dtype=float)


# -----------------------------------------
# Weight matrices
# -----------------------------------------

W_Q = np.array([
    [1, 0],
    [0, 1],
    [1, 0],
    [0, 1]
], dtype=float)

W_K = np.array([
    [1, 0],
    [0, 1],
    [1, 0],
    [0, 1]
], dtype=float)

W_V = np.array([
    [1, 0],
    [0, 1],
    [1, 1],
    [1, 1]
], dtype=float)


# -----------------------------------------
# Create Q K V
# -----------------------------------------

Q = X @ W_Q
K = X @ W_K
V = X @ W_V


# -----------------------------------------
# Attention scores
# -----------------------------------------

scores = Q @ K.T


# -----------------------------------------
# Scaling
# -----------------------------------------

d_k = K.shape[-1]

scores = scores / np.sqrt(d_k)


# -----------------------------------------
# Softmax
# -----------------------------------------

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


attention = softmax(scores)


# -----------------------------------------
# Final output
# -----------------------------------------

output = attention @ V


# -----------------------------------------
# Display
# -----------------------------------------

print("WORDS:")
print(words)

print("\nQ:")
print(Q)

print("\nK:")
print(K)

print("\nV:")
print(V)

print("\nAttention Scores:")
print(scores)

print("\nAttention Weights:")
print(attention)

print("\nFinal Output:")
print(output)

WORDS:
['The', 'cat', 'sat']

Q:
[[2. 0.]
 [0. 2.]
 [2. 2.]]

K:
[[2. 0.]
 [0. 2.]
 [2. 2.]]

V:
[[2. 1.]
 [1. 2.]
 [3. 3.]]

Attention Scores:
[[2.82842712 0.         2.82842712]
 [0.         2.82842712 2.82842712]
 [2.82842712 2.82842712 5.65685425]]

Attention Weights:
[[0.48564771 0.02870457 0.48564771]
 [0.02870457 0.48564771 0.48564771]
 [0.05285739 0.05285739 0.89428521]]

Final Output:
[[2.45694314 2.        ]
 [2.         2.45694314]
 [2.84142782 2.84142782]]


---

# 10. Understanding each Q, K, V

This is the part you should put in your notes.

| Component     | Meaning                            | Purpose                           |
| :------------ | :--------------------------------- | :-------------------------------- |
| **Q — Query** | What am I looking for?             | Searches for relevant information |
| **K — Key**   | What information do I contain?     | Used for matching                 |
| **V — Value** | What information should I provide? | Carries actual information        |

Think:

```text
Q → Search
K → Match
V → Information
```

or even simpler:

```text
Query → Question
Key   → Label
Value → Content
```

---

# 11. PyTorch implementation

Now let's implement the same thing using PyTorch.

In [3]:
import torch
import torch.nn.functional as F

# ------------------------------------
# Input
# ------------------------------------

X = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 1.0, 1.0]
])

print("X:")
print(X)


# ------------------------------------
# Dimensions
# ------------------------------------

d_model = 4
d_k = 2


# ------------------------------------
# Learnable projection matrices
# ------------------------------------

W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)


# ------------------------------------
# Q K V
# ------------------------------------

Q = X @ W_Q
K = X @ W_K
V = X @ W_V


print("\nQ:")
print(Q)

print("\nK:")
print(K)

print("\nV:")
print(V)


# ------------------------------------
# Attention scores
# ------------------------------------

scores = Q @ K.T


# ------------------------------------
# Scaling
# ------------------------------------

scores = scores / torch.sqrt(
    torch.tensor(d_k, dtype=torch.float32)
)


# ------------------------------------
# Softmax
# ------------------------------------

attention_weights = F.softmax(scores, dim=-1)


print("\nAttention Weights:")
print(attention_weights)


# ------------------------------------
# Final output
# ------------------------------------

output = attention_weights @ V

print("\nFinal Output:")
print(output)

X:
tensor([[1., 0., 1., 0.],
        [0., 1., 0., 1.],
        [1., 1., 1., 1.]])

Q:
tensor([[ 0.7638, -1.7098],
        [-2.4015,  1.3812],
        [-1.6378, -0.3285]])

K:
tensor([[-0.5380,  0.2647],
        [ 1.4053, -0.2010],
        [ 0.8673,  0.0637]])

V:
tensor([[ 0.9316, -1.0717],
        [ 3.1942, -1.6397],
        [ 4.1258, -2.7114]])

Attention Weights:
tensor([[0.1144, 0.5739, 0.3117],
        [0.9100, 0.0213, 0.0688],
        [0.7558, 0.0887, 0.1555]])

Final Output:
tensor([[ 3.2257, -1.9087],
        [ 1.1994, -1.1965],
        [ 1.6291, -1.3771]])


---

# 12. Important: Q, K, V are learned

In the examples above, we manually created:

```python
W_Q
W_K
W_V
```

But in a real Transformer, these are learned during training.

Initially:

```text
WQ → random
WK → random
WV → random
```

During training:

```text
Prediction
    ↓
Loss
    ↓
Backpropagation
    ↓
Update WQ
Update WK
Update WV
```

Eventually the model learns useful transformations.

So:

```text
Input X

      ↓

┌───────────────┐
│     WQ        │ → Query representation
├───────────────┤
│     WK        │ → Key representation
├───────────────┤
│     WV        │ → Value representation
└───────────────┘

      ↓

Attention
```

---

# 13. Real Transformer implementation

In practice, you normally don't manually create three matrices and calculate everything yourself. PyTorch provides:

```python
torch.nn.MultiheadAttention
```

Example:

In [4]:
import torch
import torch.nn as nn

# ------------------------------------
# Parameters
# ------------------------------------

d_model = 8
num_heads = 2
seq_length = 4
batch_size = 1


# ------------------------------------
# Input embeddings
# ------------------------------------

X = torch.randn(
    batch_size,
    seq_length,
    d_model
)


# ------------------------------------
# Multi-Head Attention
# ------------------------------------

attention = nn.MultiheadAttention(
    embed_dim=d_model,
    num_heads=num_heads,
    batch_first=True
)


# ------------------------------------
# Self Attention
# ------------------------------------

output, weights = attention(
    X,   # Query
    X,   # Key
    X    # Value
)


print("Input shape:")
print(X.shape)

print("\nOutput shape:")
print(output.shape)

print("\nAttention weights:")
print(weights.shape)

print("\nOutput:")
print(output)

Input shape:
torch.Size([1, 4, 8])

Output shape:
torch.Size([1, 4, 8])

Attention weights:
torch.Size([1, 4, 4])

Output:
tensor([[[ 0.1320,  0.0482, -0.1686, -0.2801,  0.2504,  0.1582,  0.3060,
           0.0275],
         [ 0.0568,  0.2259, -0.0048, -0.1071,  0.3719,  0.0614,  0.0298,
          -0.2034],
         [ 0.0508,  0.1830, -0.0171, -0.1691,  0.2058,  0.0480, -0.0058,
          -0.1465],
         [ 0.0886,  0.0625, -0.0457, -0.2883, -0.0613,  0.0020,  0.2637,
           0.0082]]], grad_fn=<TransposeBackward0>)


Notice this:

```python
attention(X, X, X)
```

That's why it's called **self-attention**.

The same sequence provides:

```text
Q ← X
K ← X
V ← X
```

But internally the layer transforms them using different learned projections.

---

# 14. Self-attention vs cross-attention

This will also make your previous question much clearer.

### Self-attention

```python
attention(X, X, X)
```

Therefore:

```text
Q ← X
K ← X
V ← X
```

Example:

```text
"The cat is sleeping"

The ↔ cat ↔ is ↔ sleeping
```

Every token can attend to other tokens in the same sequence.

---

### Cross-attention

Suppose an encoder produces:

```text
Encoder output
       ↓
    H_encoder
```

and decoder has:

```text
Decoder output
       ↓
    H_decoder
```

Then:

```text
Q ← Decoder
K ← Encoder
V ← Encoder
```

```python
attention(
    decoder,
    encoder,
    encoder
)
```

So:

```text
Self Attention:

Q ← X
K ← X
V ← X


Cross Attention:

Q ← Decoder
K ← Encoder
V ← Encoder
```

---

# 15. Complete Transformer-style implementation from scratch

Here is a more advanced program that you should study after the basic NumPy program.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SelfAttention(nn.Module):

    def __init__(self, d_model, d_k):

        super().__init__()

        self.W_Q = nn.Linear(d_model, d_k)
        self.W_K = nn.Linear(d_model, d_k)
        self.W_V = nn.Linear(d_model, d_k)


    def forward(self, X):

        # -----------------------------
        # 1. Create Q K V
        # -----------------------------

        Q = self.W_Q(X)
        K = self.W_K(X)
        V = self.W_V(X)


        # -----------------------------
        # 2. Calculate QK^T
        # -----------------------------

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )


        # -----------------------------
        # 3. Scale
        # -----------------------------

        d_k = K.size(-1)

        scores = scores / (d_k ** 0.5)


        # -----------------------------
        # 4. Softmax
        # -----------------------------

        attention_weights = F.softmax(
            scores,
            dim=-1
        )


        # -----------------------------
        # 5. Weighted sum
        # -----------------------------

        output = torch.matmul(
            attention_weights,
            V
        )


        return output, attention_weights


# =====================================
# Test
# =====================================

X = torch.randn(
    1,       # batch
    5,       # sequence length
    8        # embedding dimension
)


model = SelfAttention(
    d_model=8,
    d_k=4
)


output, weights = model(X)


print("Input:")
print(X.shape)

print("\nAttention weights:")
print(weights.shape)

print("\nOutput:")
print(output.shape)

Input:
torch.Size([1, 5, 8])

Attention weights:
torch.Size([1, 5, 5])

Output:
torch.Size([1, 5, 4])


Output shapes:

```text
Input:
torch.Size([1, 5, 8])

Attention weights:
torch.Size([1, 5, 5])

Output:
torch.Size([1, 5, 4])
```

The important part is:

```text
Input
1 × 5 × 8

      ↓

Q
1 × 5 × 4

K
1 × 5 × 4

V
1 × 5 × 4

      ↓

QKᵀ

1 × 5 × 5

      ↓

Attention Matrix

1 × 5 × 5

      ↓

Attention × V

1 × 5 × 4
```

---

# 16. One very important concept

Don't think:

> "Q, K, V are three copies of the input."

Instead think:

> **Q, K and V are three learned views of the same input.**

For self-attention:

```text
                 SAME INPUT
                     X
              ┌──────┼──────┐
              ↓      ↓      ↓
             WQ     WK     WV
              ↓      ↓      ↓
              Q      K      V
              │      │      │
              └──┬───┘      │
                 ↓          │
                QKᵀ         │
                 ↓          │
              Softmax       │
                 ↓          │
             Attention      │
               Weights      │
                 │          │
                 └────┬─────┘
                      ↓
                  Output
```

### The entire mechanism in one line:

$$ Output =
softmax\left(
\frac{(XW_Q)(XW_K)^T}{\sqrt{d_k}}
\right)(XW_V)
$$

If you understand this equation and the NumPy program above, you understand the **core mathematical mechanism behind Transformer self-attention**.

### Recommended learning order

Since you're learning LLMs from the basics toward advanced topics, I would study the programs in this order:

```text
1. Basic dot-product attention
        ↓
2. Q, K, V from scratch
        ↓
3. Scaled dot-product attention
        ↓
4. Self-attention
        ↓
5. Attention with padding mask
        ↓
6. Causal / masked self-attention
        ↓
7. Multi-head attention
        ↓
8. Cross-attention
        ↓
9. Positional encoding
        ↓
10. Complete Transformer
        ↓
11. GPT-style decoder
        ↓
12. LLM from scratch
```

The **next most useful program** is a **numerical self-attention example where we manually calculate Q, K, V, `QKᵀ`, scaling, softmax/α values, and the final output for every word**, so you can see exactly where every number comes from.